# W18 小組：資料清洗與管線建置（資料工程師 DE）

- 負責人：F114118124（認領資料工程師 DE）｜主題：金融信用違約預測（小組版）
- 輸入：`input/raw.csv`（主題原始資料，與個人 W12 同一份 credit-dirty.csv，219 列）
- 輸出：`output/cleaned.csv`，交給 BA、DS、ML、Visual、QA 使用
- 依據：個人 W12 向 Client／PM／QA 確認的欄位語意（ID 不可補號、late＝已逾期月數未知、AGE -1／unknown 來源、BILL 空白＝新戶、n/a＝同步失敗、01/02/2024＝MM/DD 台北時間），以及小組版的交付規格：`cleaned.csv` 必須保留 8 個原始欄位、至少 50 列，且非受限／非洩漏欄位的缺值比例 ≤ 5%。
- 與個人 W12 的差異：個人 W12 把 late 與 n/a 設 NULL；小組版下游要直接建模，缺值上限 5%，所以改成「保守補值＋旗標欄」，讓下游知道哪些值是補的。

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_SEED = 42  # 本 notebook 沒有隨機步驟，固定 seed 供日後抽樣沿用
np.random.seed(RANDOM_SEED)
Path("output").mkdir(exist_ok=True)

raw = pd.read_csv("input/raw.csv", dtype=str, keep_default_na=False)
print("讀入列數：", len(raw))
print("欄位：", list(raw.columns))

## 處理策略與影響筆數

| 步驟 | 欄位 | 策略 | 理由 |
|---|---|---|---|
| 1 | ID | 空白排除；完全相同的重複列保留第一筆 | ID 是之後與還款紀錄對帳的申請編號，不能補號 |
| 2 | AGE | -1、unknown、空白、超出 18–100 設 NULL | 三種寫法都是「不知道年齡」；AGE 為受限特徵，只保留欄位不供建模 |
| 3 | MARITAL_STATUS | ??、NULL 設 NULL，其餘保留 | 拒答；受限特徵，只因交付規格保留欄位 |
| 4 | PAY_0 | late 補 1，並新增 `PAY_0_LATE_UNKNOWN`=1 | late 只知道「至少逾期 1 個月」，1 是已知下限；不補 -2（未消費）以免把最高風險的人改成最低風險；旗標保留「月數未知」資訊 |
| 5 | BILL_AMT1 | 空白補 0；n/a 補有帳單者中位數，並新增 `BILL_AMT1_IMPUTED`=1 | 空白是新戶未出帳，金額真的是 0；n/a 是同步失敗，金額存在但未知，用中位數並標記 |
| 6 | signup_at | 統一成 ISO-8601 UTC；not-a-date 維持 NULL | 無法還原的時間不臆測；signup_at 不作為建模特徵 |
| 7 | default | 轉整數 0／1 | 目標變數 |
| 8 | LEAK_FUTURE_DEFAULT | 原樣保留欄位 | 與 default 完全相同屬資料洩漏；只因交付規格保留，下游由系統剔除，不得作為特徵 |

下一格實際執行並印出每一步影響筆數。

In [ ]:
df = raw.copy()
log = []

blank = df["ID"].str.strip() == ""
rejected_blank = df[blank].assign(reason="ID 空白，無法與還款紀錄對帳")
df = df[~blank]
dup = df.duplicated(keep="first")
rejected_dup = df[dup].assign(reason="與同 ID 列完全相同的重複")
df = df[~dup].copy()
assert df["ID"].is_unique, "同 ID 內容不同的重複列需要人工判斷"
log.append(("ID 空白排除", int(blank.sum())))
log.append(("完全重複隔離", int(dup.sum())))

age = pd.to_numeric(df["AGE"], errors="coerce")
age = age.where(age.between(18, 100))
log.append(("AGE 設 NULL", int(age.isna().sum())))
df["AGE"] = age.astype("Int64")

ms = df["MARITAL_STATUS"].where(df["MARITAL_STATUS"].isin(["single", "married", "divorced"]))
log.append(("MARITAL_STATUS 設 NULL", int(ms.isna().sum())))
df["MARITAL_STATUS"] = ms

late = df["PAY_0"].str.strip().str.lower() == "late"
pay = pd.to_numeric(df["PAY_0"].mask(late, "1"), errors="coerce")
df["PAY_0"] = pay.astype("Int64")
df["PAY_0_LATE_UNKNOWN"] = late.astype(int)
log.append(("PAY_0 late 補 1（旗標）", int(late.sum())))

bill_raw = df["BILL_AMT1"].str.strip()
bill = pd.to_numeric(bill_raw, errors="coerce")
new_acct = bill_raw == ""
sync_fail = bill_raw.str.lower() == "n/a"
median_bill = float(bill[bill > 0].median())
bill = bill.mask(new_acct, 0.0).mask(sync_fail, median_bill)
df["BILL_AMT1"] = bill
df["BILL_AMT1_IMPUTED"] = sync_fail.astype(int)
log.append(("BILL_AMT1 空白補 0（新戶）", int(new_acct.sum())))
log.append((f"BILL_AMT1 n/a 補中位數 {median_bill:.0f}（旗標）", int(sync_fail.sum())))

def to_utc_iso(s):
    s = (s or "").strip()
    for fmt in ("%Y-%m-%dT%H:%M:%SZ", "%Y/%m/%d %H:%M%z"):
        t = pd.to_datetime(s, format=fmt, errors="coerce", utc=True)
        if not pd.isna(t):
            return t.strftime("%Y-%m-%dT%H:%M:%SZ")
    t = pd.to_datetime(s, format="%m/%d/%Y", errors="coerce")
    if not pd.isna(t):
        return t.tz_localize("Asia/Taipei").tz_convert("UTC").strftime("%Y-%m-%dT%H:%M:%SZ")
    return None

df["signup_at"] = df["signup_at"].map(to_utc_iso)
log.append(("signup_at 無法解析維持 NULL", int(df["signup_at"].isna().sum())))
df["default"] = pd.to_numeric(df["default"], errors="coerce").astype("Int64")
df["LEAK_FUTURE_DEFAULT"] = pd.to_numeric(df["LEAK_FUTURE_DEFAULT"], errors="coerce").astype("Int64")

for step, n in log:
    print(f"{step}：{n} 列")
print("清洗後列數：", len(df))

## 拒絕列與原因

被排除的列不會進入 `cleaned.csv`，另存 `output/rejected_rows.csv` 供稽核。

In [ ]:
rejected = pd.concat([rejected_blank, rejected_dup], ignore_index=True)
print(rejected["reason"].value_counts().to_string())
print("拒絕列合計：", len(rejected), "／原始", len(raw), "（", round(len(rejected) / len(raw) * 100, 1), "%）")
rejected.to_csv("output/rejected_rows.csv", index=False)

In [ ]:
cols = ["ID", "AGE", "MARITAL_STATUS", "PAY_0", "BILL_AMT1", "default", "signup_at", "LEAK_FUTURE_DEFAULT",
        "PAY_0_LATE_UNKNOWN", "BILL_AMT1_IMPUTED"]
out = df[cols]
counted = ["ID", "PAY_0", "BILL_AMT1", "default", "signup_at"]
core_rate = out[counted].isna().to_numpy().mean()
print("只計原始 5 欄（ID、PAY_0、BILL_AMT1、default、signup_at）的缺值比例：", round(core_rate * 100, 2), "%")
system_cols = [c for c in out.columns if c not in ("AGE", "MARITAL_STATUS", "LEAK_FUTURE_DEFAULT")]
print("依系統規則（排除受限與洩漏欄、其餘欄位全計，含兩個旗標欄）的缺值比例：", round(out[system_cols].isna().to_numpy().mean() * 100, 2), "%")
print("各欄空值：", out.isna().sum().to_dict())
assert out["ID"].notna().all() and out["ID"].is_unique
assert out["PAY_0"].dropna().between(-2, 9).all()
assert set(out["default"].dropna().unique()) <= {0, 1}
out.to_csv("output/cleaned.csv", index=False)
print("已寫出 output/cleaned.csv：", len(out), "列")

## 給下游的限制與假設

- **不得當特徵的欄位**：AGE、MARITAL_STATUS（法遵禁用，SOW 1.1）、LEAK_FUTURE_DEFAULT（與 default 完全相同的未來資訊）。這三欄只因交付規格保留在檔案裡。
- **補值的欄位要看旗標**：`PAY_0_LATE_UNKNOWN`=1 的列，PAY_0 的 1 是「至少逾期 1 個月」的下限，真實逾期可能更久；這群人在個人 W13 分析中違約率最高（約 48%），建模時建議把旗標一起當特徵。`BILL_AMT1_IMPUTED`=1 的列，帳單金額是中位數，不是真實值。
- **BILL_AMT1 = 0 是新戶**，不是低消費；與真正低帳單的人意義不同。
- **signup_at** 只有兩種有效值（只反映舊系統格式），另有 NULL，不建議當特徵或用來切時間。
- **樣本量**：清洗後 190 列，各分群樣本少，下游報告的分群數字應附樣本數。
- **假設**：01/02/2024 依 Client 說明視為 2024-01-02 台北時間 00:00；若日後確認為 DD/MM，需重算 signup_at。
- **缺值比例的誠實說明**：缺值全部來自 signup_at 的 50 筆 not-a-date（無法還原，不臆測）。只計原始 5 欄時為 5.26%，略高於小組規格 5%；系統規則把兩個旗標欄也算進分母，結果約 3.76%。旗標欄是為了讓下游知道哪些值是補的，不是為了壓低比例；若教師認為應只計原始欄位，這份資料在 signup_at 上不符規格，需要 Client 補正申請時間。